In [24]:
import logging
import datetime
import numpy as np
from pathlib import Path
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from keras import Sequential, Input, Model
from keras.layers import Dense
from keras.models import load_model
from sktime.classification.deep_learning import (
    InceptionTimeClassifier,
    CNNClassifier,
    LSTMFCNClassifier,
    ResNetClassifier,
)
from sktime.classification.kernel_based import RocketClassifier
from sktime.classification.shapelet_based import ShapeletTransformClassifier
from sklearn.metrics import roc_auc_score, classification_report

from src.module.utils import load_npy, load_config

In [19]:
def balance_dataset(X, y):
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    if len(pos_idx) > len(neg_idx):
        pos_idx = np.random.choice(pos_idx, size=len(neg_idx), replace=False)
    else:
        neg_idx = np.random.choice(neg_idx, size=len(pos_idx), replace=False)

    pos_X, pos_y = X[pos_idx], y[pos_idx]
    neg_X, neg_y = X[neg_idx], y[neg_idx]
    res_X = np.vstack([pos_X, neg_X])
    res_y = np.hstack([pos_y, neg_y])
    print(res_y.shape)
    logging.info(
        f"X: {res_X.shape} / y(pos/neg): {res_y.size}({sum(res_y==1)}/{sum(res_y==0)})"
    )
    return res_X, res_y


def init_callbacks(save_path) -> list:
    callbacks = [
        ModelCheckpoint(
            save_path,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_auc", factor=0.1, patience=10, mode="max", verbose=1
        ),
        EarlyStopping(
            monitor="val_auc", patience=15, mode="max", start_from_epoch=15, verbose=1
        ),
    ]
    return callbacks


def replace_head(model, input_shape, n_classes) -> Model:
    keras_model = model.build_model(input_shape=input_shape, n_classes=n_classes)
    x = keras_model.layers[-2].output
    output = Dense(1, activation="sigmoid")(x)
    new_model = Model(inputs=keras_model.input, outputs=output)
    new_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["auc"])
    return new_model


def train_model(model, train_X, train_y, val_X, val_y, callbacks):
    print(model.summary())
    history = model.fit(
        train_X,
        train_y,
        validation_data=(val_X, val_y),
        epochs=1000,
        batch_size=64,
        callbacks=callbacks,
        verbose=2,
    )
    return model, history


def evaluate_model(model, test_X, test_y, threshold):
    prob_y = model.predict(test_X)
    pred_y = np.where(prob_y >= threshold, 1, 0)
    print("AUROC: ", roc_auc_score(test_y, prob_y))
    print(classification_report(test_y, pred_y))

In [7]:
# model_name = "rocket"
model_name = "inceptiontime"
strategy = "hypophetversion2"
src_path = Path("../../data/vitaldb/06.train_val_test_new")

In [25]:
config = load_config("../../config/conf.yaml")

In [8]:
train_X = np.load(src_path / f"vitaldb_{strategy}_train_X.npy")
train_y = np.load(src_path / f"vitaldb_{strategy}_train_y.npy")
val_X = np.load(src_path / f"vitaldb_{strategy}_val_X.npy")
val_y = np.load(src_path / f"vitaldb_{strategy}_val_y.npy")
test_X = np.load(src_path / f"vitaldb_{strategy}_test_X.npy")
test_y = np.load(src_path / f"vitaldb_{strategy}_test_y.npy")

In [9]:
train_X.shape, val_X.shape

((145146, 3000, 1), (37335, 3000, 1))

In [10]:
(train_y == 1).sum(), (train_y == 0).sum()

(49724, 95422)

In [11]:
(val_y == 1).sum(), (val_y == 0).sum()

(13407, 23928)

In [20]:
train_bal_X, train_bal_y = balance_dataset(train_X, train_y)
print(train_bal_X.shape, train_bal_y.shape)

(99448,)
(99448, 3000, 1) (99448,)


In [21]:
save_dir = Path("../../model")
save_dt = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
save_path = save_dir / (save_dt + f"_{strategy}") / f"{model_name}.model.keras"

In [22]:
callbacks = init_callbacks(save_path)

In [26]:
# clf = CNNClassifier(kernel_size=7, batch_size=32, n_conv_layers=2, random_state=42)
# clf = LSTMFCNClassifier(
#     dropout=0.8, kernel_sizes=(8, 5, 3), filter_sizes=(128, 256, 128), lstm_size=8
# )
# clf = ResNetClassifier(random_state=42)
# clf = RocketClassifier(
#     num_kernels=10000,
#     rocket_transform='rocket',
#     max_dilations_per_kernel=32,
#     n_features_per_kernel=4,
#     random_state=42,
# )
clf = InceptionTimeClassifier(
    kernel_size=40,
    n_filters=32,
    use_residual=True,
    use_bottleneck=True,
    bottleneck_size=32,
    depth=3,
    random_state=config["seed"],
)

In [27]:
if model_name in ["inceptiontime", "cnn", "lstmfcn", "resnet"]:
    clf = replace_head(clf, train_bal_X.shape[1:], 2)

2026-10-05 17:32:05.723950: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Max
2026-10-05 17:32:05.723989: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-10-05 17:32:05.723999: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-10-05 17:32:05.724034: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-05 17:32:05.724046: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [14]:
if model_name in ['inceptiontime', 'cnn', 'lstmfcn', 'resnet']:
    clf, history = train_model(clf, train_bal_X, train_bal_y, val_X, val_y, callbacks)
else:
    clf.fit(train_bal_X, train_bal_y)

In [15]:
# load model
# loaded = load_model(
#     "../../model/260320_001228_hypophetversion2/inceptiontime.model.keras"
# )

In [16]:
evaluate_model(clf, test_X, test_y, 0.5)

AUROC:  0.5
              precision    recall  f1-score   support

         0.0       0.54      1.00      0.70      8052
         1.0       0.00      0.00      0.00      6914

    accuracy                           0.54     14966
   macro avg       0.27      0.50      0.35     14966
weighted avg       0.29      0.54      0.38     14966



c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\spass-paper-3HAXnoOl-py3.11\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\spass-paper-3HAXnoOl-py3.11\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\spass-paper-3HAXnoOl-py3.11\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samp